# 03 실전 반도체 공정 데이터 분석 · 11. AI 활용

- 강의 페이지: `Web/강좌/03_실전_반도체_공정_데이터분석/실전_반도체_공정_데이터분석_강의자료.html` → **6 AI 활용** 탭
- 앞 수업에서 배운 내용을 AI로 다시 확인하고, 헷갈린 부분은 AI 연습 문제로 채우고, 배운 코드를 응용해 봅니다.
- `fab.csv`가 이 노트북과 같은 폴더에 있어야 합니다.
- 위에서 아래로 순서대로 실행하세요. 다른 노트북의 변수를 사용하지 않으므로 이 파일만 열어도 실행됩니다.

## AI 사용 전 3가지 약속

AI는 정답을 대신 내 주는 도구가 아니라, 내가 이해했는지 확인해 주는 학습 도우미입니다.

1. 먼저 스스로 풀어 보고, 막힐 때 AI에게 묻습니다.
2. AI의 설명이나 코드는 노트북에서 **직접 실행해 확인**합니다. 실행 결과가 AI의 말보다 우선입니다.
3. 회사 데이터·장비명·Lot 번호는 넣지 않고, 수업용 예시 데이터만 사용합니다.

프롬프트의 `[대괄호]` 부분만 내 상황에 맞게 바꿔서 사용하세요.

### 준비 · 1~8단계 코드 실행 (모델 학습까지)

앞 단계 코드를 그대로 모아 한 번에 실행합니다. 출력은 앞 노트북과 같습니다. `Pass_Fail`은 -1 = 정상, 1 = 불량이고, `y`는 불량을 1로 바꾼 값입니다.

In [ ]:
# ── 1단계 ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif   # ⭐ NEW
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# 그래프 한글 설정 (Mac은 'AppleGothic', Colab·리눅스는 'NanumGothic')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# ── 2단계 ──
df = pd.read_csv('fab.csv')

# ── 3단계 결측값 처리 ──
# 1️⃣ 컬럼별 결측률 계산
miss_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(miss_pct.sort_values(ascending=False).head(10))

# 2️⃣ 결측률 50% 초과 컬럼 제거
THRESHOLD = 50.0
cols_to_drop = miss_pct[miss_pct > THRESHOLD].index.tolist()
print(f"🗑️ 제거할 컬럼: {len(cols_to_drop)}개")
df = df.drop(columns=cols_to_drop)

# 3️⃣ 남은 결측치는 중앙값으로 채우기 (타겟 제외!)
numeric_cols = df.select_dtypes(include='number').columns.tolist()
numeric_cols.remove('Pass_Fail')   # ⚠️ 타겟은 절대 채우지 말 것!

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
print(f"✅ 결측 처리 완료. 남은 결측: {df.isnull().sum().sum()}")

# ── 4단계 분산 0 제거 ──
# 분산 = 0 컬럼 + 거의 0인 컬럼 모두 제거
variances = df[numeric_cols].var()

constant_cols    = variances[variances == 0].index.tolist()
near_constant    = variances[(variances > 0) & (variances < 1e-6)].index.tolist()

print(f"분산 0  컬럼: {len(constant_cols)}개")
print(f"분산 ≈0 컬럼: {len(near_constant)}개")

to_drop_var = constant_cols + near_constant
df = df.drop(columns=to_drop_var)
numeric_cols = [c for c in numeric_cols if c not in to_drop_var]

print(f"✅ 사용 가능한 센서: {len(numeric_cols)}개")

# ── 5단계 특성 선택 ──
# 타겟을 0/1로 변환: 불량(1)을 양성 클래스로
y = (df['Pass_Fail'] == 1).astype(int)
X_all = df[numeric_cols].copy()

# 상위 K=20 개 센서 자동 선택
K = 20
selector = SelectKBest(score_func=f_classif, k=K)
selector.fit(X_all, y)

f_scores   = pd.Series(selector.scores_, index=numeric_cols)\
               .replace([np.inf, -np.inf], np.nan).dropna()
top_k_cols = f_scores.sort_values(ascending=False).head(K).index.tolist()

print(f"🏆 선택된 상위 {K}개 센서:")
for i, col in enumerate(top_k_cols, 1):
    print(f"  {i:2d}. {col}  F={f_scores[col]:6.1f}")

# ── 7단계 분리 & 스케일링 ──
X = df[top_k_cols].copy()    # 상위 K개 센서만!
# y = (df['Pass_Fail'] == 1).astype(int)  # 위에서 이미 만듦

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # ⭐ 불균형 비율 유지 필수!
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"학습: {X_train.shape}, 그 중 불량 {y_train.sum()}건")
print(f"테스트: {X_test.shape}, 그 중 불량 {y_test.sum()}건")

# ── 8단계 모델 학습 ──
# 1) 로지스틱 회귀
lr_model = LogisticRegression(
    random_state=42, max_iter=1000,
    class_weight='balanced'   # ⭐ 소수 클래스에 가중치
)
lr_model.fit(X_train_scaled, y_train)

# 2) 랜덤 포레스트
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    class_weight='balanced'
)
rf_model.fit(X_train_scaled, y_train)

# 예측
y_pred_lr = lr_model.predict(X_test_scaled)
y_pred_rf = rf_model.predict(X_test_scaled)

## 1. 배운 내용 AI로 다시 확인하기

수업에서 배운 함수나 개념 하나를 골라 쉬운 설명과 작은 예제를 받아 봅니다.

| 수업 | 다시 확인해 볼 함수·개념 |
|---|---|
| 1 데이터 탐색 | read_csv(), shape, info(verbose=False), value_counts() — 클래스 불균형 |
| 2 데이터 정제 | 결측률, drop(columns=), median()으로 채우기, 분산 0 컬럼 제거 |
| 3 특성 선택 | SelectKBest(f_classif), F-점수, 박스플롯, 상관관계 히트맵 |
| 4 모델링 | train_test_split(stratify), StandardScaler, class_weight='balanced', 혼동행렬, ROC |
| 5 보강 실습 | 기준 모델, TN·FP·FN·TP, 임계값, 모델 비교, 데이터 누수 |

### 개념 다시 설명받기

```text
나는 반도체 공정 데이터로 하는 머신러닝 분류(scikit-learn)을(를) 처음 배우는 초보자입니다.
수업에서 [함수나 개념 이름, 예: train_test_split()의 stratify]을(를) 배웠는데 아직 잘 이해가 안 됩니다.

1. 이것이 무엇을 하는지 중학생도 이해할 수 있게 두세 문장으로 설명해 주세요.
2. 5줄 이하의 아주 작은 예제 코드와, 실행하면 나올 결과를 보여 주세요.
3. 초보자가 자주 하는 실수 1가지를 알려 주세요.
어려운 용어를 쓸 때는 괄호 안에 쉬운 말로 풀어 주세요.
```

**AI 설명 확인하는 방법**

1. AI가 보여 준 예제 코드를 아래 빈 셀에 붙여 넣고 실행합니다.
2. AI가 말한 결과와 실제 실행 결과가 같은지 비교합니다.
3. 다르면 실행 결과를 그대로 AI에게 보여 주고 왜 다른지 다시 물어봅니다.

In [ ]:
# AI가 보여 준 예제 코드를 여기에 붙여 넣고 실행해 보세요.


### 내 말로 설명하고 점검받기

노트북을 보지 않고 먼저 내 말로 적어 본 뒤 AI에게 점검받으세요.

```text
내가 [함수나 개념 이름]을(를) 내 말로 설명해 볼게요.
틀린 부분이나 빠진 부분이 있는지 확인해 주세요.
바로 고쳐 쓰지 말고, 어디가 부족한지 힌트를 먼저 주세요.

내 설명: [여기에 내 말로 쓴 설명]
```

## 2. 헷갈린 부분 AI 연습 문제로 채우기

수업에서 부족했던 부분을 스스로 찾고, AI에게 그 부분만 연습 문제를 만들어 달라고 합니다.

### ① 자가 점검 — 자신 없는 항목을 1~2개 고르세요

- [ ] 정확도가 높아도 좋은 모델이 아닐 수 있는 이유를 설명할 수 있다
- [ ] `train_test_split()`에서 `stratify=y`를 쓰는 이유를 설명할 수 있다
- [ ] 혼동행렬에서 FN(불량을 정상으로 놓친 수)을 찾을 수 있다
- [ ] Recall과 Precision의 차이를 설명할 수 있다
- [ ] `fit_transform()`과 `transform()`의 차이(데이터 누수)를 설명할 수 있다

### ② 맞춤 연습 문제 요청

```text
나는 반도체 공정 데이터로 하는 머신러닝 분류(scikit-learn)을(를) 배우는 초보자입니다.
[자신 없는 부분, 예: 혼동행렬에서 FN(놓친 불량)과 Recall 읽기]이(가) 헷갈립니다.
이 부분만 연습할 수 있는 문제를 3개 만들어 주세요.

규칙:
1. 쉬운 문제 → 조금 어려운 문제 순서로 내 주세요.
2. 한 번에 한 문제씩만 내고, 내가 답을 보낼 때까지 정답을 보여 주지 마세요.
3. 문제에 필요한 데이터는 5행 이하의 작은 표를 만드는 코드로 함께 주세요. (정상 0 / 불량 1 라벨과 센서 2~3개 정도의 예시)
4. pandas와 scikit-learn(train_test_split, StandardScaler, LogisticRegression, RandomForestClassifier, confusion_matrix, recall_score)만 사용하게 해 주세요.
5. 내가 틀리면 정답을 바로 주지 말고 힌트를 한 번 먼저 주세요.
```

AI가 낸 문제를 아래 셀에서 한 문제씩 풀어 보세요. 문제에 나온 데이터 만들기 코드도 함께 붙여 넣습니다.

In [ ]:
# AI 연습 문제 1 풀이


In [ ]:
# AI 연습 문제 2 풀이


In [ ]:
# AI 연습 문제 3 풀이


### ③ 채점과 오답 정리

```text
방금 푼 문제의 내 답과 실행 결과입니다. 채점해 주세요.

[내 코드]
[실행 결과 또는 오류 메시지]

1. 맞았는지 먼저 말해 주세요.
2. 틀렸다면 어느 줄이 왜 틀렸는지 쉬운 말로 설명해 주세요.
3. 마지막에 '오답 노트'로 한 줄 정리해 주세요. (예: 여러 조건은 각각 괄호로 감싼다)
```

<details>
<summary><strong>오류가 나서 막혔을 때</strong></summary>

```text
아래 코드를 실행했더니 오류가 났습니다. 정답 코드를 바로 주지 말고
1) 오류 메시지가 무슨 뜻인지, 2) 어느 줄을 먼저 살펴보면 되는지 알려 주세요.

[코드]
[오류 메시지 전체]
```

</details>

### 📝 오답 노트

이 셀을 더블클릭해 직접 채워 보세요. 다음 수업 전에 같은 프롬프트로 문제를 다시 받아 풀어 보면 복습이 됩니다.

| 헷갈린 부분 | 틀린 이유 | 기억할 한 줄 |
|---|---|---|
|  |  |  |
|  |  |  |

## 3. 배운 코드로 응용해 보기

새 문법을 배우지 않고, 수업 코드를 조금 바꿔 새로운 질문에 답해 봅니다.
먼저 직접 작성하고, 막히면 **힌트 → AI 힌트 → 정답 예제** 순서로 확인하세요.

### 응용 1 · 센서 수를 20개에서 10개로 줄이면?

5단계 **특성 선택**과 7~9단계 **분리·스케일링·Recall**을 다시 사용합니다.

**목표:** F-점수 상위 10개 센서만으로 로지스틱 회귀를 다시 학습하고, 20개일 때와 불량 Recall을 비교합니다.

In [ ]:
# TODO 1: top_k_cols[:10]으로 상위 10개 센서 이름을 top10_cols에 저장하세요.
# TODO 2: 7단계와 같은 방법(test_size=0.2, random_state=42, stratify=y)으로 나누고 스케일링하세요.
# TODO 3: 로지스틱 회귀(class_weight='balanced')를 학습하고 20개일 때와 불량 Recall을 비교하세요.


<details>
<summary><strong>힌트 보기</strong></summary>

7단계 코드에서 `df[top_k_cols]`를 `df[top10_cols]`로 바꾸면 됩니다. 스케일러는 학습 데이터에만 `fit_transform()`, 테스트 데이터에는 `transform()`만 씁니다. Recall은 `recall_score(정답, 예측)`으로 구합니다.

</details>

<details>
<summary><strong>AI에게 힌트만 받기</strong></summary>

```text
[F-점수 상위 10개 센서만으로 로지스틱 회귀를 다시 학습하고, 20개일 때와 불량 Recall을 비교합니다.]을 풀고 있습니다. 정답 코드는 주지 말고,
어떤 순서로 풀면 되는지 단계만 3개 이내로 알려 주세요.
pandas와 scikit-learn(train_test_split, StandardScaler, LogisticRegression, RandomForestClassifier, confusion_matrix, recall_score) 범위에서 힌트를 주세요.
```

</details>

**정답 예제:** 먼저 직접 시도한 뒤 아래 셀을 실행하세요.

In [ ]:
from sklearn.metrics import recall_score

top10_cols = top_k_cols[:10]
X10_train, X10_test, y10_train, y10_test = train_test_split(
    df[top10_cols], y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler10 = StandardScaler()
X10_train_scaled = scaler10.fit_transform(X10_train)   # 학습 데이터에만 fit
X10_test_scaled = scaler10.transform(X10_test)         # 테스트는 transform만

lr10 = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr10.fit(X10_train_scaled, y10_train)

recall_20 = recall_score(y_test, y_pred_lr)
recall_10 = recall_score(y10_test, lr10.predict(X10_test_scaled))
print(f"센서 20개 로지스틱 회귀 불량 Recall: {recall_20:.3f}")
print(f"센서 10개 로지스틱 회귀 불량 Recall: {recall_10:.3f}")

### 응용 2 · 임계값을 0.3으로 낮추면 놓친 불량이 줄어들까?

보강 실습의 **임계값**과 **혼동행렬 숫자 읽기**를 로지스틱 회귀에 적용합니다.

**목표:** 로지스틱 회귀의 불량 확률로 임계값 0.5와 0.3을 비교하고, 놓친 불량(FN)과 오탐(FP)이 어떻게 바뀌는지 확인합니다.

In [ ]:
# TODO 1: lr_model.predict_proba()로 테스트 데이터의 불량 확률을 구하세요.
# TODO 2: 임계값 0.5와 0.3마다 확률 >= 임계값이면 1로 바꾼 예측값을 만드세요.
# TODO 3: 각 예측값의 혼동행렬에서 FN, FP, TP를 꺼내 출력하세요.


<details>
<summary><strong>힌트 보기</strong></summary>

불량 확률은 `predict_proba(X_test_scaled)[:, 1]`입니다. `confusion_matrix(y_test, 예측).ravel()`은 tn, fp, fn, tp 순서로 값을 돌려줍니다.

</details>

<details>
<summary><strong>AI에게 힌트만 받기</strong></summary>

```text
[로지스틱 회귀의 불량 확률로 임계값 0.5와 0.3을 비교하고, 놓친 불량(FN)과 오탐(FP)이 어떻게 바뀌는지 확인합니다.]을 풀고 있습니다. 정답 코드는 주지 말고,
어떤 순서로 풀면 되는지 단계만 3개 이내로 알려 주세요.
pandas와 scikit-learn(train_test_split, StandardScaler, LogisticRegression, RandomForestClassifier, confusion_matrix, recall_score) 범위에서 힌트를 주세요.
```

</details>

**정답 예제:** 먼저 직접 시도한 뒤 아래 셀을 실행하세요.

In [ ]:
lr_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

for threshold in [0.5, 0.3]:
    pred = (lr_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    print(f"임계값 {threshold}: 놓친 불량(FN) {fn}건, 오탐(FP) {fp}건, 잡은 불량(TP) {tp}건")

print('임계값은 테스트 데이터가 아니라 별도 검증 데이터에서 정해야 합니다.')

### 한 걸음 더 · 비슷한 응용 문제 받기

```text
방금 [응용 과제 내용]을 풀었습니다. 내 코드는 아래와 같습니다.
[내 코드]

같은 데이터로 조금만 바꿔 볼 수 있는 응용 문제를 1개 내 주세요.
정답은 내가 요청할 때만 보여 주세요.
```

## AI 답을 믿기 전에

- [ ] AI가 만든 코드를 직접 실행해 결과를 확인했는가?
- [ ] 열 이름과 값의 의미(Pass_Fail -1 = 정상, 1 = 불량)가 수업 데이터와 같은가?
- [ ] 이해하지 못한 코드를 그대로 쓰지 않고 쉬운 설명을 다시 요청했는가?
- [ ] 회사 데이터나 실제 Lot 번호를 AI에 넣지 않았는가?

실제 회사 데이터, 장비명, 레시피값, Lot 식별자는 외부 AI 서비스에 직접 붙여 넣지 않습니다.

## 마무리

모델 결과는 항상 **혼동행렬과 불량 Recall**로 확인하세요. AI에게 문제를 받을 때도 '정확도만 보면 안 되는 이유'를 함께 설명해 달라고 요청하면 좋습니다.